DSC540-T301 <br>
Term Project: Milestone 4<br>
Jeremy Barton <br>

# Milestone 4 - Connecting to an API/Pulling in the Data and Cleaning/Formatting

USASpending.gov provides access to APIs that show grants, awards, and other funding to states and agencies in the USA. In the project, this data is joined with the wage data to understand whether or not pay is steadily increasing compared to the increase in state funding.

## Extraction

### API Extraction of Award Spending Data

In [1]:
# Imports and Preprocessing
import requests
import pandas as pd
import time
from bs4 import BeautifulSoup

url = "https://api.usaspending.gov/api/v2/search/spending_by_award/"
start_date = "2024-01-01"
end_date = "2024-12-31"

"Award Amount" represents the dollar amount and "Place of Performance State Code" will provide the best clue to determine which state was awarded.

In [2]:
# Define fields and award types
fields = [
    "Award Amount", "Total Outlays", "Award ID",
    "Recipient Name", "Recipient State Code", 
    "Awarding Agency", "Award Type",
    "Start Date", "End Date",
    "Place of Performance State Code",
    "Last Modified Date",
    
    # Additional
    "COVID-19 Obligations", "Infrastructure Obligations",
    "Awarding Sub Agency", "Funding Agency",
    "Description", "NAICS", "CFDA Number"
]

# Separate award type groups
award_groups = {
    "contracts": ["A", "B", "C", "D"],
    "grants": ["02", "03", "04", "05"],
    "loans": ["07", "08"],
    "other": ["06", "09", "10", "11", "-1"]
}

# NY-NJ-CT filter
target_states = ["CT"
#, "NJ", "NY"
]

These are the definitions for award types.<br>

<b>Contract Types</b>

A = Definitive Contract (BPA)<br>
B = Purchase Order<br>
C = Delivery Order<br>
D = Definitive Contract<br>

<b>Grant Types</b>

02 = Block Grant<br>
03 = Formula Grant<br>
04 = Project Grant<br>
05 = Cooperative Agreement<br>

<b>Loan Types</b>

07 = Direct Loan<br>
08 = Guaranteed/Insured Loan

Creating the payload to attach to the reuqest using POST.

Begin fetching the data from the USASpending API.

The request orders the return of results by 100 through different pages.
Results are stored to a variable called `spending_data`. For Milestone 2, only the contracts from Connecticut will be pulled and cleaned. This same process will be used for the `NY` and `NJ` data.

NOTE TO SELF: Running the below takes about 5 minutes to complete.

In [3]:
# Fetch all award types for NY-NJ-CT
spending_data = []  # Initialize empty list to store all records from API

# Loop through each award type group (contracts, grants, loans, other)
for group_name, award_types in award_groups.items():
    print(f"\n{'='*60}")
    print(f"Fetching {group_name.upper()}...")  # Show which group is being fetched
    print(f"{'='*60}")
    
    page = 1  # Start at page 1 for each award type group
    
    # Loop through all pages of results
    while True:
        # Building the API request payload
        payload = {
            "filters": {
                # Filter by range, award type code and states NY, NJ and CT 
                # (just CT for Milestone 2)
                "time_period": [{"start_date": start_date, "end_date": end_date}],
                "award_type_codes": award_types,
                "recipient_locations": [
                    {"country": "USA", "state": state} for state in target_states
                ]
            },
            # Specify which data fields to return, page number
            #    and api_limit page (limit is 100)
            "fields": fields,
            "page": page,
            "limit": 100
        }
        
        # Make POST request to USAspending API
        response = requests.post(url, json=payload)
        
        # Check if request was successful, parse the results
        #   and save it to the running results

        if response.status_code == 200:
            data = response.json()
            results = data.get('results', [])
            
            # If no results returned, we've reached the end
            if not results:
                break
            
            # This is where we actually extend the spending_data
            #   variable with the running results
            spending_data.extend(results)

            # Show progress
            print(f"Page {page}: {len(spending_data):,} total records")
            
            # Check if there are more pages available
            if not data.get('page_metadata', {}).get('hasNext', False):
                break
            
            # Move to next page
            page += 1
            # Wait 0.3 seconds between requests
            #   for safety reasons
            time.sleep(0.3)
        else:
            # If request failed, print error and stop
            print(f"Error {response.status_code}: {response.text}")
            break

# Final summary after all award type groups are fetched
print(f"\n{'='*60}")
print(f"Total records collected: {len(spending_data):,}")
print(f"{'='*60}")


Fetching CONTRACTS...
Page 1: 100 total records
Page 2: 200 total records
Page 3: 300 total records
Page 4: 400 total records
Page 5: 500 total records
Page 6: 600 total records
Page 7: 700 total records
Page 8: 800 total records
Page 9: 900 total records
Page 10: 1,000 total records
Page 11: 1,100 total records
Page 12: 1,200 total records
Page 13: 1,300 total records
Page 14: 1,400 total records
Page 15: 1,500 total records
Page 16: 1,600 total records
Page 17: 1,700 total records
Page 18: 1,800 total records
Page 19: 1,900 total records
Page 20: 2,000 total records
Page 21: 2,100 total records
Page 22: 2,200 total records
Page 23: 2,300 total records
Page 24: 2,400 total records
Page 25: 2,500 total records
Page 26: 2,600 total records
Page 27: 2,700 total records
Page 28: 2,800 total records
Page 29: 2,900 total records
Page 30: 3,000 total records
Page 31: 3,100 total records
Page 32: 3,200 total records
Page 33: 3,300 total records
Page 34: 3,400 total records
Page 35: 3,500 tot

### Save to DataFrame

In [4]:
spending_df = pd.DataFrame(spending_data)
# Checking if the field we filtered the data on,
#   "Last Modified Date" is in the dataframe.


In [5]:
spending_df['Place of Performance State Code'] \
    .value_counts() \
    .reset_index(name='count') \
    .rename(columns={'index': 'Place of Performance State Code'}).head()

,Place of Performance State Code,count
0,CT,16233
1,IL,407
2,DC,291
3,MD,176
4,MA,163


Analyze results. Since what's being pulled is all that was <b>received</b>, state codes other than CT are listed. `Place of Performane Code` is a good measurement becuase it shows where the service was executed.

In [37]:
# Display the filter parameters used
print("")
print("DATA COLLECTION FILTERS")
print("")
start_date = spending_df['Last Modified Date'].min()
end_date   = spending_df['Last Modified Date'].max()
print(f"Date Range: {start_date} to {end_date}")
print(f"Target States: {', '.join(target_states)}")
print(f"Award Groups: {', '.join(award_groups.keys())}")
print(f"Total Records Collected: {len(spending_df):,}")


DATA COLLECTION FILTERS

Date Range: 2024-01-02 to 2026-01-30
Target States: CT
Award Groups: contracts, grants, loans, other
Total Records Collected: 19,065


According to the above analysis, the earliest date for an award is 2024-01-02 as expected from our filters. The most recent modified date is January 1st 2026 as of Febuary 1st 2026 (today).

## Cleaning

<b>`spending_df`</b>

Below is a list of the retained columns from the spending dataframe.

#### 1. Filtering

In [ ]:
s_retained = [
    'internal_id',                    # Unique internal identifier for the record in the dataset
    'Award Amount',                   # Total monetary value of the award granted
    'Award ID',                        # Government-assigned unique identifier for the award
    'Recipient Name',                  # Name of the individual, company, or organization receiving the award
    'Recipient State Code',            # Two-letter US state code for the recipient’s location
    'Awarding Agency',                 # Federal agency that issued the award
    'Start Date',                      # Date when the award or project period begins
    'End Date',                        # Date when the award or project period ends
    'Place of Performance State Code', # Two-letter US state code where the award’s work will be performed
    'Last Modified Date',              # Most recent date when the award record was updated
    'Awarding Sub Agency',             # Subdivision or office within the awarding agency responsible for the award
    'Funding Agency',                  # Agency providing the funds, may differ from the awarding agency
    'Description',                     # Text description of the purpose, scope, or use of the award
    'NAICS'                            # North American Industry Classification System code 
                                       #    describing the type of business or industry receiving the award
]


spending_df = spending_df[s_retained]

In [7]:
spending_df.head(3)

,internal_id,Award Amount,Total Outlays,Award ID,Recipient Name,Recipient State Code,Awarding Agency,Award Type,Start Date,End Date,...,COVID-19 Obligations,Infrastructure Obligations,Awarding Sub Agency,Funding Agency,Description,NAICS,CFDA Number,awarding_agency_id,agency_slug,generated_internal_id
0,309506838,3.465196e+10,52471.00,N0002417C2100,ELECTRIC BOAT CORPORATION,None,Department of Defense,None,2017-02-14,2036-06-02,...,0.0,0.0,Department of the Navy,Department of Defense,SSN 802 AND 803 LONG LEAD TIME MATERIAL,"{'code': '336611', 'description': 'SHIP BUILDI...",None,1173.0,department-of-defense,CONT_AWD_N0002417C2100_9700_-NONE-_-NONE-
1,309506843,2.431381e+10,43723356.73,N0002417C2117,ELECTRIC BOAT CORPORATION,None,Department of Defense,None,2017-09-21,2031-12-31,...,0.0,0.0,Department of the Navy,Department of Defense,COLUMBIA CLASS DESIGN COMPLETION,"{'code': '541330', 'description': 'ENGINEERING...",None,1173.0,department-of-defense,CONT_AWD_N0002417C2117_9700_-NONE-_-NONE-
2,309505151,1.961875e+10,107066.00,N0002412C2115,ELECTRIC BOAT CORPORATION,None,Department of Defense,None,2012-04-13,2026-01-31,...,0.0,0.0,Department of the Navy,Department of Defense,SSN 792 LONG LEAD TIME MATERIAL,"{'code': '336611', 'description': 'SHIP BUILDI...",None,1173.0,department-of-defense,CONT_AWD_N0002412C2115_9700_-NONE-_-NONE-


#### 2. Searching for Duplicates

In [13]:
spending_df.duplicated()

TypeError: unhashable type: 'dict'

Okay, so it appears we can't find dupes because one or more of the columns contains Python dictionary.

Lets some investigating. 🔎

In [14]:
spending_df.applymap(type).eq(dict).any()

/tmp/ipykernel_29383/794981906.py:1: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  spending_df.applymap(type).eq(dict).any()


internal_id                        False
Award Amount                       False
Total Outlays                      False
Award ID                           False
Recipient Name                     False
Recipient State Code               False
Awarding Agency                    False
Award Type                         False
Start Date                         False
End Date                           False
Place of Performance State Code    False
Last Modified Date                 False
COVID-19 Obligations               False
Infrastructure Obligations         False
Awarding Sub Agency                False
Funding Agency                     False
Description                        False
NAICS                               True
CFDA Number                        False
awarding_agency_id                 False
agency_slug                        False
generated_internal_id              False
dtype: bool

Ah-ha! It is the "NAICS" column that is a dict.

#### 3. Convert dictionary to tuple

In [ ]:
# Using a lamdba function!
spending_df = spending_df.map(
    lambda x: tuple(sorted(x.items())) if isinstance(x, dict) else x
)

Re-trying the duplicate locate.

In [17]:
duplicates = spending_df[spending_df.duplicated()]
duplicates

,internal_id,Award Amount,Total Outlays,Award ID,Recipient Name,Recipient State Code,Awarding Agency,Award Type,Start Date,End Date,...,COVID-19 Obligations,Infrastructure Obligations,Awarding Sub Agency,Funding Agency,Description,NAICS,CFDA Number,awarding_agency_id,agency_slug,generated_internal_id


No duplcates is nice.

In [71]:
spending_df.dtypes

internal_id                                 int64
Award Amount                              float64
Award ID                                      str
Recipient Name                                str
Recipient State Code                       object
Awarding Agency                               str
Start Date                         datetime64[us]
End Date                           datetime64[us]
Place of Performance State Code               str
Last Modified Date                 datetime64[us]
Awarding Sub Agency                           str
Funding Agency                                str
Description                                   str
NAICS                                      object
dtype: object

#### 4. Trimming

Wherever there is the word "Corporation" will be replaced with "Corp." to save horizontal space.


In [20]:
spending_df = spending_df.replace("CORPORATION", "CORP.", regex=True)

#### 5. Replacing

To update for current events, the name "Department of Defense" will be changed to "Department of War".

In [22]:
spending_df = spending_df.replace("Department of Defense", "Department of War.", regex=True)

### Final Pit-Stop

Exporing the final results to check that all changes were made.

In [23]:
# Spending data
spending_df.head(3)

,internal_id,Award Amount,Total Outlays,Award ID,Recipient Name,Recipient State Code,Awarding Agency,Award Type,Start Date,End Date,...,COVID-19 Obligations,Infrastructure Obligations,Awarding Sub Agency,Funding Agency,Description,NAICS,CFDA Number,awarding_agency_id,agency_slug,generated_internal_id
0,309506838,3.465196e+10,52471.00,N0002417C2100,ELECTRIC BOAT CORP.,None,Department of War.,None,2017-02-14,2036-06-02,...,0.0,0.0,Department of the Navy,Department of War.,SSN 802 AND 803 LONG LEAD TIME MATERIAL,"((code, 336611), (description, SHIP BUILDING A...",None,1173.0,department-of-defense,CONT_AWD_N0002417C2100_9700_-NONE-_-NONE-
1,309506843,2.431381e+10,43723356.73,N0002417C2117,ELECTRIC BOAT CORP.,None,Department of War.,None,2017-09-21,2031-12-31,...,0.0,0.0,Department of the Navy,Department of War.,COLUMBIA CLASS DESIGN COMPLETION,"((code, 541330), (description, ENGINEERING SER...",None,1173.0,department-of-defense,CONT_AWD_N0002417C2117_9700_-NONE-_-NONE-
2,309505151,1.961875e+10,107066.00,N0002412C2115,ELECTRIC BOAT CORP.,None,Department of War.,None,2012-04-13,2026-01-31,...,0.0,0.0,Department of the Navy,Department of War.,SSN 792 LONG LEAD TIME MATERIAL,"((code, 336611), (description, SHIP BUILDING A...",None,1173.0,department-of-defense,CONT_AWD_N0002412C2115_9700_-NONE-_-NONE-


In [24]:
spending_df.dtypes

internal_id                          int64
Award Amount                       float64
Total Outlays                      float64
Award ID                            object
Recipient Name                      object
Recipient State Code                object
Awarding Agency                     object
Award Type                          object
Start Date                          object
End Date                            object
Place of Performance State Code     object
Last Modified Date                  object
COVID-19 Obligations               float64
Infrastructure Obligations         float64
Awarding Sub Agency                 object
Funding Agency                      object
Description                         object
NAICS                               object
CFDA Number                         object
awarding_agency_id                 float64
agency_slug                         object
generated_internal_id               object
dtype: object

In [25]:
spending_df.shape

(19145, 22)

### Ethical Implications

In this notebook we made 5 cleaning/formatting changes. Filtering out fields that won't be used in the analysis, trimming "CORPORATION" to "CORP.", searching for duplicates and updating the Department of Defense to reflect the officially reinstated title.

Regulatory guidelines for this source are similar, if not identical to that of the U.S Treasury data under the <a href="https://fiscal.treasury.gov/foia.html">Freedom of Information Act"</a>.

Risks created based on the transformations include incorrecly identifing a Recipient Name. Confusion can be avoided by only show the data from this notebook to internal teams.

This data was ethically sourced and verified, as it is from a credible government institution.

Any ethical implications identifed can be traced back to the source, identified, analyzed and answered for by refernecing the Privacy Policy and Terms of Use.

https://www.usaspending.gov/about/privacy 

https://www.treasurydirect.gov/legal-information/terms/

https://fiscal.treasury.gov/foia.html